# Lesson 4 - RAG Data Freshness

This notebook shows how CDC keeps retrieval context fresh after source data changes, then routes the same retrieval path through a LangChain agent. It uses a small in-memory context store, not a hosted vector database.

> Synthetic training data only; not for clinical use.

### Prerequisites

Run setup from the repository root:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r exercises/shared/requirements.txt
```

Each lesson runs independently using only the scripts in `exercises/shared/cdc_lessons`.

### Model API key (choose Gemini or OpenAI)

1. Sign in to [Google AI Studio](https://aistudio.google.com/apikey) and accept the terms if prompted.
2. Copy the default API key, or click **Create API key** and select a project. If your project is missing, import it from **Dashboard > Projects** first.
3. In `exercises/shared/.env` (create it from `exercises/shared/.env.example` if needed), set `GOOGLE_API_KEY=your_key_here`. Keep the key out of notebook cells and Git.
4. Restart the kernel and run all cells to load the key for the LangChain agent.

Using OpenAI or Microsoft Foundry instead? Set `LLM_PROVIDER=openai`, `OPENAI_API_KEY`, `OPENAI_MODEL` (your deployment name on Foundry) and, for Foundry only, `OPENAI_BASE_URL=https://<resource>.openai.azure.com/openai/v1/` in the same `.env` file.

See Google's [API key guide](https://ai.google.dev/gemini-api/docs/api-key) for details.


In [ ]:
import pathlib
import sys

_SHARED = next(
    (candidate / "shared" for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     for candidate in (parent, parent / "exercises")
     if (candidate / "shared" / "cdc_lessons").is_dir()),
    None,
)
if _SHARED is None:
    raise RuntimeError("Keep the shared folder alongside the Lesson 4 and 5 folders.")
if str(_SHARED) not in sys.path:
    sys.path.insert(0, str(_SHARED))

import os
from dotenv import load_dotenv
from sqlalchemy import select
from langchain.agents import create_agent
from langchain_core.tools import tool

from cdc_lessons.context_store import (
    build_context_store,
    evaluate_freshness,
    refresh_entry,
    retrieve_and_answer,
)
from cdc_lessons.models import CanonicalEvent, EventProcessingLog, PatientTimeline
from cdc_lessons.runtime import create_lesson_db, update_task

load_dotenv(_SHARED / ".env")
db = create_lesson_db()
print("Created an isolated in-memory lesson database.")


Created an isolated in-memory lesson database.


## (a) Bootstrap data and build retrieval context

`create_lesson_db()` seeds synthetic data and processes simulated CDC events into `care_work_queue`. `build_context_store` creates one bounded context entry per patient from the exercise projection.


In [ ]:
# Build a bounded context store for a couple of patients
store = build_context_store(db, patient_ids=["p2", "p3"])
print(f"Context entries built: {list(store.entries)}")

entry_p2 = store.get("p2")
print("\nExample context entry for p2:")
print(f"  evidence: {entry_p2.evidence}")
print(f"  source_record_id: {entry_p2.source_record_id}")
print(f"  source_table:     {entry_p2.source_table}")
print(f"  operation:        {entry_p2.operation}")
print(f"  source_commit_at: {entry_p2.source_commit_at}")
print(f"  context_updated_at: {entry_p2.context_updated_at}")


Context entries built: ['p2', 'p3']

Example context entry for p2:
  evidence: Morgan Lee (patient_id=p2) has priority score 50: Open task overdue (+50). Open tasks: 1 (overdue 1). Pending referrals: 0.
  source_record_id: p2
  source_table:     patients
  operation:        c
  source_commit_at: 2026-10-07 07:20:42.984116+00:00
  context_updated_at: 2026-10-07 07:20:43.002173+00:00


## (b) Retrieve and answer

`retrieve_and_answer` looks up context, checks the freshness budget, and returns evidence IDs with the answer.


In [ ]:
question = "What is the current priority for this patient?"
baseline = retrieve_and_answer(db, store, "p2", question, budget_seconds=300)

print(f"Question:      {baseline.question}")
print(f"Answer:        {baseline.answer}")
print(f"Evidence IDs:  {baseline.evidence_ids}")
print(f"Authoritative: {baseline.authoritative}")


Question:      What is the current priority for this patient?
Answer:        Morgan Lee (patient_id=p2) has priority score 50: Open task overdue (+50). Open tasks: 1 (overdue 1). Pending referrals: 0.
Evidence IDs:  ['p2']
Authoritative: True


## (b1) LangChain agent

The agent uses one retrieval tool backed by `retrieve_and_answer`; model settings come from `.env`.


In [ ]:
@tool
def retrieve_patient_context(patient_id: str, question: str) -> dict:
    """Retrieve one patient's bounded, freshness-checked context."""
    result = retrieve_and_answer(db, store, patient_id, question, budget_seconds=300)
    return {
        "answer": result.answer if result.authoritative else "Withheld: refresh required.",
        "authoritative": result.authoritative,
        "evidence_ids": result.evidence_ids,
        "freshness": {
            "decision": result.freshness.decision,
            "context_age_seconds": round(result.freshness.context_age_seconds, 1),
        },
    }

if os.getenv("LLM_PROVIDER", "gemini").lower() == "openai":
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(
        model=os.getenv("OPENAI_MODEL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL") or None,  # set only for Microsoft Foundry
        temperature=0,
        max_tokens=512,
        timeout=120,
        max_retries=2,
    )
else:
    from langchain_google_genai import ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(
        model=os.getenv("GEMINI_MODEL"),
        google_api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"),
        temperature=0,
        max_output_tokens=512,
        timeout=120,
        max_retries=2,
    )
agent = create_agent(
    model=model,
    tools=[retrieve_patient_context],
    system_prompt="Call the retrieval tool once. Answer in at most three sentences using only its result. Include evidence IDs and freshness. If authoritative is false, say a refresh is required.",
)

agent_response = agent.invoke({
    "messages": [{"role": "user", "content": f"Patient ID: p2\n{question}"}]
}, config={"recursion_limit": 6})
print(agent_response["messages"][-1].content)


[{'type': 'text', 'text': 'Patient Morgan Lee (p2) currently has a priority score of 50 due to one open overdue task, with zero pending referrals.'}, {'type': 'text', 'text': ' This data is supported by evidence ID p2 with a freshness context age of 11.3 seconds (decision: allow).', 'extras': {'signature': 'Er8GCrwGAWkUfRMGogFpBsfKW0VP+uaJx/R1/lbeIGFmJx/So6FR2Fv2z/zeVnifncd9g0jep6f7LI2BHF1f6KzqaVj8M4mFgSHyUWy1d6aXWMAOAIl3+HhvdusU4n1O9t39frCObvcUQnQWWbceA++y7yO03oiVIu8jyfV779cbm9xGpT140P79UGPpYjCU7mqaX+mTAj2MvpM2zKtQVmHuZAjwsb6JD9bGcf3A6S8cp+wqm0Z8BOGLjKV/r/CDUwHd1M5y0hBZO37/qxhe+MJtS7KpZo4SrwzrsmFdi8OpDHORQdynDENkSvMgbf1umZTKCNYz3703H9Gh1CmzFu1qMCPvyDppqzGb+hIlT4kui5GVdDmNYQ0ZWBX46F/qVUqFh0YGDcEr7dolZETRuecZ8VnMmHRXz7MQaEICq/nq83MMyUA3VN5H7WzBuvTb3YBTbUkZAdzS3Fy+4ZH1uhEjs3VIx0hhtYYhkZYn7Gh6uxop1tDJE007Lmqn/N+B65iN6CZkg8INvxfT+hqfyr5tlZEFGRNV+S7giybiOg7cc/aTVqn9Vi2Rnmnxk99VuDlfj32OldSwccJ6IZIHMLnW0eua9h+50UOInWmm+UWcXS/UqDCaQjZulQ+H+6zHFkOv4Ij7WCEhbm/vTDQuCbg7GRs017L4kMhzTtXFSPP2QKpWwZe

## (c) Baseline freshness fields

The answer includes source age, source-to-context lag, and context age. `age_seconds` remains an alias for `context_age_seconds`, which is used by the freshness budget.


In [ ]:
baseline_freshness = baseline.freshness

print(f"source_record_id:               {baseline_freshness.source_record_id}")
print(f"operation:                       {baseline_freshness.operation}")
print(f"source_commit_at:                {baseline_freshness.source_commit_at}")
print(f"context_updated_at:              {baseline_freshness.context_updated_at}")
print(f"retrieval_at:                    {baseline_freshness.retrieval_at}")
print(f"source_age_seconds:              {baseline_freshness.source_age_seconds:.3f}")
print(f"source_to_context_lag_seconds:   {baseline_freshness.source_to_context_lag_seconds:.3f}")
print(f"context_age_seconds:             {baseline_freshness.context_age_seconds:.3f}")
print(f"decision:                        {baseline_freshness.decision}")


source_record_id:               p2
operation:                       c
source_commit_at:                2026-10-07 07:20:42.984116+00:00
context_updated_at:              2026-10-07 07:20:43.002173+00:00
retrieval_at:                    2026-10-07 07:20:43.014686+00:00
source_age_seconds:              0.031
source_to_context_lag_seconds:   0.018
context_age_seconds:             0.013
decision:                        allow


## (d) Mutate source data

Call the shared exercise helper `update_task` to complete p2's overdue task and process its simulated CDC event.


In [ ]:
from cdc_lessons.models import CareWorkQueue

before_row = db.get(CareWorkQueue, "p2")
print(f"Priority BEFORE mutation: {before_row.priority_score} ({before_row.priority_reasons_json})")

correlation_id = update_task(db, "task1", status="completed")
print(f"\nCompleted task1 for p2 via update_task(); correlation_id={correlation_id}")

after_row = db.get(CareWorkQueue, "p2")
print(f"Priority AFTER mutation:  {after_row.priority_score} ({after_row.priority_reasons_json})")


Priority BEFORE mutation: 50 (["Open task overdue (+50)"])

Completed task1 for p2 via update_task(); correlation_id=0105d1a5-3afa-44a9-8d8b-a6fe534c814d
Priority AFTER mutation:  0 ([])


## (e) Inspect the CDC outcome

The write becomes a raw CDC event, a `CanonicalEvent`, a projection update, an `EventProcessingLog` record, and a `PatientTimeline` entry.


In [ ]:
latest_event = db.scalar(
    select(CanonicalEvent)
    .where(CanonicalEvent.patient_id == "p2")
    .order_by(CanonicalEvent.captured_at.desc())
)
print("Latest CanonicalEvent for p2:")
print(f"  event_id:         {latest_event.event_id}")
print(f"  source_table:     {latest_event.source_table}")
print(f"  operation:        {latest_event.operation}")
print(f"  status:           {latest_event.status}")
print(f"  source_commit_at: {latest_event.source_commit_at}")
print(f"  captured_at:      {latest_event.captured_at}")

log_entry = db.scalar(
    select(EventProcessingLog)
    .where(EventProcessingLog.event_id == latest_event.event_id)
    .order_by(EventProcessingLog.processed_at.desc())
)
print("\nConsumer outcome (EventProcessingLog):")
print(f"  outcome: {log_entry.outcome}")
print(f"  message: {log_entry.message}")

timeline_entry = db.scalar(
    select(PatientTimeline)
    .where(PatientTimeline.event_id == latest_event.event_id)
)
print("\nCDC-derived PatientTimeline entry:")
print(f"  event_type: {timeline_entry.event_type}")
print(f"  summary:    {timeline_entry.summary}")


Latest CanonicalEvent for p2:
  event_id:         2e97e91e-7f76-4cd3-86d9-ba82899d13b3
  source_table:     care_tasks
  operation:        u
  status:           processed
  source_commit_at: 2026-10-07 07:21:35.948009
  captured_at:      2026-10-07 07:21:35.948009

Consumer outcome (EventProcessingLog):
  outcome: processed
  message: Updated care_work_queue and patient_timeline

CDC-derived PatientTimeline entry:
  event_type: care_tasks.u
  summary:    care_tasks changed; priority is now 0


## (f) Refresh only the affected context entry

Only the changed patient entry is rebuilt. The p3 entry is unchanged because p3 was not affected.


In [ ]:
stale_entry_p2 = entry_p2  # keep a handle to the pre-refresh entry for the comparison below
untouched_p3_before = store.get("p3")

refreshed_entry_p2 = refresh_entry(db, store, "p2")

print(f"p2 context_updated_at BEFORE refresh: {stale_entry_p2.context_updated_at}")
print(f"p2 context_updated_at AFTER refresh:  {refreshed_entry_p2.context_updated_at}")
print(f"p3 entry unchanged by targeted refresh: {store.get('p3') is untouched_p3_before}")


p2 context_updated_at BEFORE refresh: 2026-10-07 07:20:43.002173+00:00
p2 context_updated_at AFTER refresh:  2026-10-07 07:21:35.973500+00:00
p3 entry unchanged by targeted refresh: True


## (g) Compare stale and refreshed answers

`stale_answer` uses the old context entry; `refreshed_answer` uses the updated one.


In [ ]:
stale_freshness = evaluate_freshness(stale_entry_p2, budget_seconds=300)
stale_answer = (
    stale_entry_p2.evidence
    if stale_freshness.decision == "allow"
    else f"(withheld: {stale_freshness.reason})"
)

refreshed_answer = retrieve_and_answer(db, store, "p2", question, budget_seconds=300)

print("STALE (pre-refresh) view of p2:")
print(f"  answer:   {stale_answer}")
print(f"  facts:    priority_score={stale_entry_p2.facts['priority_score']}")
print()
print("REFRESHED (post-CDC-event) view of p2:")
print(f"  answer:        {refreshed_answer.answer}")
print(f"  evidence_ids:  {refreshed_answer.evidence_ids}")
print(f"  facts:         priority_score={refreshed_entry_p2.facts['priority_score']}")

assert stale_entry_p2.facts["priority_score"] != refreshed_entry_p2.facts["priority_score"], (
    "Expected the mutation to change the priority score reflected in the refreshed context."
)


STALE (pre-refresh) view of p2:
  answer:   Morgan Lee (patient_id=p2) has priority score 50: Open task overdue (+50). Open tasks: 1 (overdue 1). Pending referrals: 0.
  facts:    priority_score=50

REFRESHED (post-CDC-event) view of p2:
  answer:        Morgan Lee (patient_id=p2) has priority score 0: no active priority rules. Open tasks: 0 (overdue 0). Pending referrals: 0.
  evidence_ids:  ['task1']
  facts:         priority_score=0


## (h) Enforce a freshness budget

When context is older than the budget, `retrieve_and_answer` blocks the authoritative answer instead of returning stale evidence.


In [ ]:
from datetime import timedelta

blocked = retrieve_and_answer(
    db, store, "p2", question, budget_seconds=0,
    retrieval_at=store.get("p2").context_updated_at + timedelta(seconds=1),
)
print("Tiny budget (0s):")
print(f"  decision:      {blocked.freshness.decision}")
print(f"  authoritative: {blocked.authoritative}")
print(f"  answer:        {blocked.answer}")
assert blocked.freshness.decision == "block"
assert not blocked.authoritative

allowed = retrieve_and_answer(db, store, "p2", question, budget_seconds=10_000)
print("\nGenerous budget (10,000s):")
print(f"  decision:      {allowed.freshness.decision}")
print(f"  authoritative: {allowed.authoritative}")
assert allowed.freshness.decision == "allow"
assert allowed.authoritative


Tiny budget (0s):
  decision:      block
  authoritative: False
  answer:        Answer withheld pending refresh: Context age 1.0s exceeds freshness budget 0.0s. Last known evidence: Morgan Lee (patient_id=p2) has priority score 0: no active priority rules. Open tasks: 0 (overdue 0). Pending referrals: 0.

Generous budget (10,000s):
  decision:      allow
  authoritative: True


## (i) Freshness policy check

Use `evaluate_freshness` and `budget_seconds` to choose immediate refresh for urgent questions and scheduled refresh for routine reporting.


In [ ]:
def choose_freshness_policy(question: str) -> tuple[str, float]:
    """Return (policy_name, budget_seconds) for the given question."""
    urgent_markers = ("overdue", "now", "urgent", "immediately")
    if any(marker in question.lower() for marker in urgent_markers):
        return "push-now", 5.0
    return "wait-until-next-step", 300.0


for sample_question in [
    "Is this patient's overdue task fixed now?",
    "What is today's coordination queue for the daily report?",
]:
    policy_name, budget = choose_freshness_policy(sample_question)
    print(f"{sample_question!r} -> policy={policy_name}, budget_seconds={budget}")


"Is this patient's overdue task fixed now?" -> policy=push-now, budget_seconds=5.0
"What is today's coordination queue for the daily report?" -> policy=wait-until-next-step, budget_seconds=300.0


**Recap:** You built bounded retrieval context, changed source data, refreshed only the affected entry, compared stale vs. refreshed answers, and enforced a freshness budget. Continue to Lesson 5 for the governed tool gateway.


In [ ]:
db.close()